<a href="https://colab.research.google.com/github/eshaans1305/BIS_lab/blob/main/1BM24CS096_Lab3_Ant_colony.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#ANT COLONY OPTIMIZATION FOR SHORTEST PATH IN COMPUTER NETWORK

import random
import numpy as np

class AntColonyRouting:
    def __init__(self, num_routers, cost_matrix, source, destination, num_ants=10, generations=50, alpha=1.0, beta=2.0, evaporation=0.5, Q=100.0):
        self.num_routers = num_routers
        self.cost_matrix = np.array(cost_matrix)
        self.source = source
        self.destination = destination
        self.num_ants = num_ants
        self.generations = generations
        self.alpha = alpha
        self.beta = beta
        self.evaporation = evaporation
        self.Q = Q

        self.pheromone = np.ones((num_routers, num_routers))
        self.heuristic = np.zeros((num_routers, num_routers))
        for i in range(num_routers):
            for j in range(num_routers):
                if self.cost_matrix[i][j] != float('inf') and self.cost_matrix[i][j] > 0:
                    self.heuristic[i][j] = 1.0 / self.cost_matrix[i][j]
                else:
                    self.heuristic[i][j] = 0.0

    def solve(self):
        best_path = None
        best_cost = float('inf')

        for gen in range(self.generations):
            paths = []
            costs = []

            for ant in range(self.num_ants):
                path = self.find_path()
                if path:
                    cost = self.calculate_cost(path)
                    paths.append(path)
                    costs.append(cost)
                    if cost < best_cost:
                        best_cost = cost
                        best_path = path

            self.update_pheromones(paths, costs)

        return best_path, best_cost

    def find_path(self):
        path = [self.source]
        current = self.source
        visited = {self.source}

        while current != self.destination:
            neighbors = [i for i in range(self.num_routers) if self.cost_matrix[current][i] != float('inf') and i not in visited]
            if not neighbors:
                return None

            probabilities = []
            for n in neighbors:
                tau = self.pheromone[current][n] ** self.alpha
                eta = self.heuristic[current][n] ** self.beta
                probabilities.append(tau * eta)

            total = sum(probabilities)
            if total == 0:
                next_node = random.choice(neighbors)
            else:
                probabilities = [p / total for p in probabilities]
                next_node = random.choices(neighbors, weights=probabilities, k=1)[0]

            path.append(next_node)
            visited.add(next_node)
            current = next_node

        return path

    def calculate_cost(self, path):
        cost = 0
        for i in range(len(path) - 1):
            cost += self.cost_matrix[path[i]][path[i+1]]
        return cost

    def update_pheromones(self, paths, costs):
        self.pheromone *= (1.0 - self.evaporation)
        for path, cost in zip(paths, costs):
            if cost > 0:
                deposit = self.Q / cost
                for i in range(len(path) - 1):
                    self.pheromone[path[i]][path[i+1]] += deposit

if __name__ == "__main__":
    INF = float('inf')

    num_routers = 6
    cost_matrix = [
        [0, 2, 4, INF, INF, INF],
        [2, 0, 1, 4, 2, INF],
        [4, 1, 0, INF, 3, INF],
        [INF, 4, INF, 0, 3, 2],
        [INF, 2, 3, 3, 0, 2],
        [INF, INF, INF, 2, 2, 0]
    ]

    source = 0
    destination = 5

    aco = AntColonyRouting(
        num_routers=num_routers,
        cost_matrix=cost_matrix,
        source=source,
        destination=destination,
        num_ants=15,
        generations=100,
        alpha=1.0,
        beta=2.0,
        evaporation=0.4
    )

    best_path, best_cost = aco.solve()

    print("Optimization Completed Successfully.")
    print(f"Shortest Path from Router {source} to Router {destination}: {best_path}")
    print(f"Minimum Communication Cost/Delay: {best_cost}")


Optimization Completed Successfully.
Shortest Path from Router 0 to Router 5: [0, 1, 4, 5]
Minimum Communication Cost/Delay: 6.0
